# Data Cleaning and Transformation 

## Contents 

### Raw to Target Schema
- Processes raw tables into final clean versions and inserts into database
- Lists raw schema at the beginning of each table section and target at the end
- Creates two quarentine tables for medications and procedures for rows whose spans are inverted (stop < start)

### Summary
- Lists changes made and notes for each table

### Cleaning Steps From Profiling

Patients:
- Rename Id to ID
- 174 null Fips in Polk county

Encounters:
- Type-cast reasoncode to varchar 
- Type-cast start and stop from timestamp to date and rename to _date
- Rename Id to ID

Conditions: 
- Type-cast code to varchar

Observations:  
- Type-cast date from timestamp to date

Procedures: 
- Type-cast both code and reasoncode to varchar
- Procedures 296 codes to 299 descriptions implying code case differences although differences may be due to using approximate unique counts
- Type-cast start and stop from timestamp to date and rename to _date

Medications:
- Type-cast code to varchar
- 286 codes to 507 descriptions so distinct codes match to multiple descriptions implying description case differences
- Type-cast start and stop from timestamp to date and rename to _date

General
- Date ranges can run into 2027 so all filtering using dates should be bound within 2026
- Drop the 1452 patients under 65


In [1]:
%load_ext sql
%sql duckdb:///../ma_quality.duckdb
%config SqlMagic.displaylimit = 35

Connecting to 'duckdb:///../ma_quality.duckdb'

## Raw to Target Schema

### Patients

In [2]:
%%sql

SELECT * FROM raw_patients LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME
18acf270-d944-4228-ef95-e395bb4e0dba,1960-11-23,None,999-83-4252,S99959458,X89257224X,Mr.,Roberto515,Carlos172,Ceballos171,None,None,S,white,hispanic,M,Port-au-Prince Haiti HT,947 Connelly Way Unit 59,Des Moines,Iowa,Polk County,19153,50321,41.62333050695145,-93.63517328711768,673915.57,57679.03,72596
4ab6f786-0913-43c5-7121-afb3db14952a,1960-12-03,None,999-78-8439,S99919226,X79787747X,Mrs.,Lakiesha286,Elna874,Armstrong51,None,Braun514,M,white,nonhispanic,F,Des Moines Iowa US,590 Swaniawski Alley Unit 26,Des Moines,Iowa,Polk County,19153,50317,41.68010868170922,-93.64098846884534,222882.74,922733.21,26758
a96f4412-e403-f107-78ce-8ca08d2863dc,1956-10-22,None,999-64-9684,S99980541,X34462013X,Mrs.,Sherrie396,Shalon513,Dickens475,None,Brown30,W,white,nonhispanic,F,Ankeny Iowa US,787 Champlin Pathway Suite 88,Des Moines,Iowa,Polk County,19153,50319,41.56472796279022,-93.6467738199394,213694.79,283491.33,58940
7ecfde43-6f06-5da9-d6db-05fba4c24340,1951-02-26,None,999-45-9580,S99979814,X24907654X,Mrs.,Brittney657,Keith571,Schaefer657,None,Schamberger479,M,white,nonhispanic,F,Newton Iowa US,916 Rutherford Extension,Urbandale,Iowa,Polk County,19153,50325,41.73963308334789,-93.77999104786026,95182.5,432842.84,25730
726803cb-9079-e62d-ed4a-d89c5ecbb29b,1957-12-31,None,999-79-1325,S99969698,X14017534X,Ms.,Jong718,Nan75,Lind531,None,None,S,black,nonhispanic,F,Des Moines Iowa US,472 McKenzie Walk Suite 52,Des Moines,Iowa,Polk County,19153,50313,41.66064076334253,-93.65663429261129,26580.02,908802.17,20516


In [3]:
%%sql 

SELECT column_name, column_type, min, max, approx_unique, count, null_percentage
FROM (SUMMARIZE raw_patients)

Running query in 'duckdb:///../ma_quality.duckdb'

column_name,column_type,min,max,approx_unique,count,null_percentage
Id,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,30000,0.00
BIRTHDATE,DATE,1941-01-19,1962-01-15,8487,30000,0.00
DEATHDATE,VARCHAR,None,None,0,30000,100.00
SSN,VARCHAR,999-10-1016,999-99-9972,27157,30000,0.00
DRIVERS,VARCHAR,S99910004,S99999996,29053,30000,0.00
PASSPORT,VARCHAR,X10000664X,X9999051X,24890,30000,0.00
PREFIX,VARCHAR,Mr.,Ms.,3,30000,0.00
FIRST,VARCHAR,Aaron697,Óscar156,7206,30000,0.00
MIDDLE,VARCHAR,Aaron697,Óscar156,7138,30000,20.13
LAST,VARCHAR,Abbott774,Ávila241,1036,30000,0.00


The nulls in columns Middle, Suffix and Maiden will be left as is because null is the correct representation of absence and these columns will not be used to model.

Deathdate column is 100% null and will be dropped because the data was generated with the restriction that every patient is alive through the end of the generation

*Null Fips investigation*

Fips are a geographic marker of a persons state - county and should have zero nulls

In [4]:
%%sql

SELECT county, count(*) AS n
FROM raw_patients
WHERE fips IS NULL
GROUP BY county
ORDER BY n DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

COUNTY,n
Polk County,174


In [5]:
%%sql

SELECT zip, count(*) AS n
FROM raw_patients
WHERE fips IS NULL
GROUP BY zip
ORDER BY n DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

ZIP,n
00000,174


In [6]:
%%sql

SELECT county, state, city, count(*) AS n
FROM raw_patients
WHERE zip = '00000'
GROUP BY county, state, city
ORDER BY n DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

COUNTY,STATE,CITY,n
Polk County,Iowa,Saylorville,174


All 174 null Fips codes mapped to a corrupted zipcode 00000 in Saylorville, Polk County. Would have to confirm a systemmatic error from those care centers to cast to iowa - polk county fips number. For now since the count is an insignifcant proportion of the population being 174/28458 or .006% of our total population and .007% of polk county I will cast the 00000 zipcodes to null to represent the missing values while noting to look into for later.

In [7]:
%%sql

SELECT *
FROM raw_patients 
GROUP BY ALL
HAVING count(*) > 1

Running query in 'duckdb:///../ma_quality.duckdb'

Id,BIRTHDATE,DEATHDATE,SSN,DRIVERS,PASSPORT,PREFIX,FIRST,MIDDLE,LAST,SUFFIX,MAIDEN,MARITAL,RACE,ETHNICITY,GENDER,BIRTHPLACE,ADDRESS,CITY,STATE,COUNTY,FIPS,ZIP,LAT,LON,HEALTHCARE_EXPENSES,HEALTHCARE_COVERAGE,INCOME


In [8]:
%%sql

SELECT DISTINCT PREFIX FROM raw_patients

Running query in 'duckdb:///../ma_quality.duckdb'

PREFIX
Ms.
Mr.
Mrs.


In [9]:
%%sql 

SELECT DISTINCT SUFFIX FROM raw_patients

Running query in 'duckdb:///../ma_quality.duckdb'

SUFFIX
JD
PhD
MD
None


In [10]:
%%sql

SELECT DISTINCT MARITAL FROM raw_patients

Running query in 'duckdb:///../ma_quality.duckdb'

MARITAL
M
D
W
S


In [11]:
%%sql

SELECT DISTINCT RACE FROM raw_patients

Running query in 'duckdb:///../ma_quality.duckdb'

RACE
hawaiian
white
native
asian
other
black


In [12]:
%%sql

SELECT DISTINCT ETHNICITY FROM raw_patients

Running query in 'duckdb:///../ma_quality.duckdb'

ETHNICITY
nonhispanic
hispanic


In [13]:
%%sql

SELECT DISTINCT GENDER FROM raw_patients

Running query in 'duckdb:///../ma_quality.duckdb'

GENDER
M
F


In [14]:
%%sql 

DROP TABLE IF EXISTS procedures_quarantine;
DROP TABLE IF EXISTS procedures;
DROP TABLE IF EXISTS medications_quarantine;
DROP TABLE IF EXISTS medications;
DROP TABLE IF EXISTS observations;
DROP TABLE IF EXISTS conditions;
DROP TABLE IF EXISTS encounters;
DROP TABLE IF EXISTS population;
DROP TABLE IF EXISTS patients;

Running query in 'duckdb:///../ma_quality.duckdb'

Success


In [15]:
%%sql

SET TIME ZONE 'UTC';

-- PATIENTS
CREATE OR REPLACE TABLE patients (
    id         TEXT PRIMARY KEY,
	--Deathdate omitted because population was generated in a way that did not allow for death instances (100% null)
    birthdate  DATE,
    ssn        TEXT,
    drivers    TEXT,
    passport   TEXT,
    prefix     TEXT,
    first      TEXT,
    middle     TEXT,
    last       TEXT,
    suffix     TEXT,
    maiden     TEXT,
    marital    TEXT,
    race       TEXT,
    ethnicity  TEXT,
    gender     TEXT,
    birthplace TEXT,
    address    TEXT,
    city       TEXT,
    state      TEXT,
    county     TEXT,
    fips       TEXT,
    zip        TEXT,
    lat        DOUBLE PRECISION,
    lon        DOUBLE PRECISION
);

INSERT INTO patients
SELECT
    id,
    birthdate,
    ssn,
    drivers,
    passport,
    prefix,
    first,
    middle,
    last,
    suffix,
    maiden,
    marital,
    race,
    ethnicity,
    gender,
    birthplace,
    address,
    city,
    state,
    county,
    fips::TEXT,
    NULLIF(zip, '00000'),
    lat,
    lon
FROM raw_patients
;

SELECT * FROM patients LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

id,birthdate,ssn,drivers,passport,prefix,first,middle,last,suffix,maiden,marital,race,ethnicity,gender,birthplace,address,city,state,county,fips,zip,lat,lon
18acf270-d944-4228-ef95-e395bb4e0dba,1960-11-23,999-83-4252,S99959458,X89257224X,Mr.,Roberto515,Carlos172,Ceballos171,None,None,S,white,hispanic,M,Port-au-Prince Haiti HT,947 Connelly Way Unit 59,Des Moines,Iowa,Polk County,19153,50321,41.62333050695145,-93.63517328711768
4ab6f786-0913-43c5-7121-afb3db14952a,1960-12-03,999-78-8439,S99919226,X79787747X,Mrs.,Lakiesha286,Elna874,Armstrong51,None,Braun514,M,white,nonhispanic,F,Des Moines Iowa US,590 Swaniawski Alley Unit 26,Des Moines,Iowa,Polk County,19153,50317,41.68010868170922,-93.64098846884534
a96f4412-e403-f107-78ce-8ca08d2863dc,1956-10-22,999-64-9684,S99980541,X34462013X,Mrs.,Sherrie396,Shalon513,Dickens475,None,Brown30,W,white,nonhispanic,F,Ankeny Iowa US,787 Champlin Pathway Suite 88,Des Moines,Iowa,Polk County,19153,50319,41.56472796279022,-93.6467738199394
7ecfde43-6f06-5da9-d6db-05fba4c24340,1951-02-26,999-45-9580,S99979814,X24907654X,Mrs.,Brittney657,Keith571,Schaefer657,None,Schamberger479,M,white,nonhispanic,F,Newton Iowa US,916 Rutherford Extension,Urbandale,Iowa,Polk County,19153,50325,41.73963308334789,-93.77999104786026
726803cb-9079-e62d-ed4a-d89c5ecbb29b,1957-12-31,999-79-1325,S99969698,X14017534X,Ms.,Jong718,Nan75,Lind531,None,None,S,black,nonhispanic,F,Des Moines Iowa US,472 McKenzie Walk Suite 52,Des Moines,Iowa,Polk County,19153,50313,41.66064076334253,-93.65663429261129


### Encounters

In [17]:
%%sql

SELECT * FROM raw_encounters LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION
18acf270-d944-4228-2665-8fb041c9d0b8,1961-08-19 09:44:10+00:00,1961-08-19 09:59:10+00:00,18acf270-d944-4228-ef95-e395bb4e0dba,3b0513a0-490a-3696-a075-c1d3fbfb4125,8b122f44-622a-3d67-a752-9c300d9067d4,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,185347001,Encounter for problem (procedure),84.68,84.68,0.0,735029006,Shellfish (substance)
a96f4412-e403-f107-4464-59069193524a,1967-09-08 23:03:11+00:00,1967-09-08 23:30:14+00:00,a96f4412-e403-f107-78ce-8ca08d2863dc,f2bb4ca9-9e82-3b8d-be93-17fedb1ae144,7dbfe5aa-e2f6-3b14-9f3f-508689412921,e03e23c9-4df1-3eb6-a62d-f70f02301496,ambulatory,390906007,Follow-up encounter (procedure),84.68,825.08,0.0,203646004,Adolescent idiopathic scoliosis (disorder)
4ab6f786-0913-43c5-ea27-af41e8eefd15,1961-08-13 16:36:41+00:00,1961-08-13 16:51:41+00:00,4ab6f786-0913-43c5-7121-afb3db14952a,3b0513a0-490a-3696-a075-c1d3fbfb4125,8b122f44-622a-3d67-a752-9c300d9067d4,df166300-5a78-3502-a46a-832842197811,ambulatory,185347001,Encounter for problem (procedure),84.68,84.68,34.68,735029006,Shellfish (substance)
7ecfde43-6f06-5da9-765c-c14e1fc8ecd4,1968-04-15 15:29:33+00:00,1968-04-15 15:47:26+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,85841532-977c-323f-ad93-2026b45fc2a9,f2849c78-a5f8-3185-8013-2a693c05ef6a,df166300-5a78-3502-a46a-832842197811,wellness,410620009,Well child visit (procedure),77.0,583.2,433.2,None,None
726803cb-9079-e62d-335f-808659046c0e,1976-02-24 16:17:34+00:00,1976-02-24 17:15:42+00:00,726803cb-9079-e62d-ed4a-d89c5ecbb29b,21b6965f-e0b4-31b2-92a4-ec511e298d60,b0d4f1ea-ee58-3e5c-992c-0e1af9d0698a,df166300-5a78-3502-a46a-832842197811,wellness,162673000,General examination of patient (procedure),77.0,768.3,618.3,None,None


In [18]:
%%sql 

SELECT *
FROM raw_encounters
GROUP BY ALL 
HAVING count(*) > 1

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Id,START,STOP,PATIENT,ORGANIZATION,PROVIDER,PAYER,ENCOUNTERCLASS,CODE,DESCRIPTION,BASE_ENCOUNTER_COST,TOTAL_CLAIM_COST,PAYER_COVERAGE,REASONCODE,REASONDESCRIPTION


In [19]:
%%sql

SELECT column_name, column_type, min, max, approx_unique, count, null_percentage
FROM (SUMMARIZE raw_encounters)

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,count,null_percentage
Id,VARCHAR,0003b2f9-73d5-3dcd-0098-a8a6ec394d02,fff5352f-2b53-d86c-fdcf-8582350378f2,3569313,3605226,0.00
START,TIMESTAMP WITH TIME ZONE,1941-03-27 04:13:30+00,2027-02-14 23:59:20+00,3339438,3605226,0.00
STOP,TIMESTAMP WITH TIME ZONE,1941-03-27 06:13:30+00,2027-02-18 17:30:55+00,3172723,3605226,0.00
PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,3605226,0.00
ORGANIZATION,VARCHAR,000ed674-8120-3e32-80b9-116308f69349,ffd4e8c0-677b-3635-9c06-34298fcccea9,196,3605226,0.00
PROVIDER,VARCHAR,00277f12-bd33-34a7-a2e1-a79413285c24,ffd10a95-760f-38f6-8907-e42bd3f756d2,165,3605226,0.00
PAYER,VARCHAR,0133f751-9229-3cfd-815f-b6d4979bdd6a,e03e23c9-4df1-3eb6-a62d-f70f02301496,10,3605226,0.00
ENCOUNTERCLASS,VARCHAR,ambulatory,wellness,10,3605226,0.00
CODE,BIGINT,1505002,453131000124105,54,3605226,0.00
DESCRIPTION,VARCHAR,Administration of vaccine to produce active immunity (procedure),Well child visit (procedure),54,3605226,0.00


In [20]:
%%sql

SELECT
    count(*) AS total,
    count(*) FILTER (WHERE reasoncode IS NULL) AS null_code,
    count(*) FILTER (WHERE reasondescription IS NULL) AS null_desc,
    count(*) FILTER (WHERE reasoncode IS NULL AND reasondescription IS NULL) AS both_null,
    count(*) FILTER (WHERE (reasoncode IS NULL) <> (reasondescription IS NULL)) AS mismatched
FROM raw_encounters;

Running query in 'duckdb:///../ma_quality.duckdb'

total,null_code,null_desc,both_null,mismatched
3605226,1452938,1452938,1452938,0


In [21]:
%%sql

SELECT
    encounterclass,
    count(*) AS n,
    count(*) FILTER (WHERE reasoncode IS NULL) AS null_reason,
    round(100.0 * count(*) FILTER (WHERE reasoncode IS NULL) / count(*), 1) AS pct_null
FROM raw_encounters
GROUP BY encounterclass
ORDER BY n DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

ENCOUNTERCLASS,n,null_reason,pct_null
ambulatory,2032070,142187,7.0
wellness,630033,630033,100.0
outpatient,372793,355681,95.4
urgentcare,272083,266756,98.0
emergency,125857,57813,45.9
home,90071,0,0.0
inpatient,51332,0,0.0
snf,15173,0,0.0
hospice,10580,7,0.1
virtual,5234,461,8.8


Nulls in reason and reasoncode are always together and not concentrated entirely in one area. Correct representation of absence. 

In [22]:
%%sql

SELECT
    count(*) FILTER (WHERE CAST(stop AT TIME ZONE 'UTC' AS DATE)
                        <  CAST(start AT TIME ZONE 'UTC' AS DATE)) AS inverted_spans,
    count(*) FILTER (WHERE stop IS NULL) AS null_stop
FROM raw_encounters;

Running query in 'duckdb:///../ma_quality.duckdb'

inverted_spans,null_stop
0,0


In [23]:
%%sql

SELECT CAST(code AS TEXT) AS code, count(DISTINCT description) AS n_desc
FROM raw_encounters
GROUP BY code
HAVING count(DISTINCT description) > 1
ORDER BY n_desc DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

code,n_desc


In [24]:
%%sql

SELECT CAST(reasoncode AS TEXT) AS reasoncode, count(DISTINCT reasondescription) AS n_desc
FROM raw_encounters
WHERE reasoncode IS NOT NULL
GROUP BY reasoncode
HAVING count(DISTINCT reasondescription) > 1
ORDER BY n_desc DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

reasoncode,n_desc


In [25]:
%%sql

SELECT
    count(*) FILTER (WHERE total_claim_cost < 0)                    AS neg_claim,
    count(*) FILTER (WHERE payer_coverage < 0)                      AS neg_coverage,
    count(*) FILTER (WHERE payer_coverage > total_claim_cost)       AS coverage_exceeds_claim
FROM raw_encounters;

Running query in 'duckdb:///../ma_quality.duckdb'

neg_claim,neg_coverage,coverage_exceeds_claim
0,0,0


In [26]:
%%sql

SELECT DISTINCT PAYER FROM raw_encounters

Running query in 'duckdb:///../ma_quality.duckdb'

PAYER
8fa6c185-e44e-3e34-8bd8-39be8694f4ce
734afbd6-4794-363b-9bc0-6a3981533ed5
a735bf55-83e9-331a-899d-a82a60b9f60c
26aab0cd-6aba-3e1b-ac5b-05c8867e762c
e03e23c9-4df1-3eb6-a62d-f70f02301496
d31fccc3-1767-390d-966a-22a5156f4219
d18ef2e6-ef40-324c-be54-34a5ee865625
df166300-5a78-3502-a46a-832842197811
0133f751-9229-3cfd-815f-b6d4979bdd6a
b046940f-1664-3047-bca7-dfa76be352a4


In [27]:
%%sql

SELECT DISTINCT ENCOUNTERCLASS FROM raw_encounters

Running query in 'duckdb:///../ma_quality.duckdb'

ENCOUNTERCLASS
outpatient
inpatient
hospice
urgentcare
home
wellness
emergency
snf
ambulatory
virtual


In [28]:
%%sql

SELECT DISTINCT BASE_ENCOUNTER_COST, ENCOUNTERCLASS
FROM raw_encounters

Running query in 'duckdb:///../ma_quality.duckdb'

BASE_ENCOUNTER_COST,ENCOUNTERCLASS
84.68,outpatient
141.13,urgentcare
77.0,wellness
141.13,outpatient
62.76,emergency
94.16,hospice
117.82,snf
62.76,inpatient
93.53,home
104.6,emergency


In [ ]:
%%sql

CREATE OR REPLACE TABLE encounters (
    id                   TEXT PRIMARY KEY,
    patient              TEXT NOT NULL REFERENCES patients(id),
    start_date           DATE NOT NULL,
    stop_date            DATE NOT NULL,
    organization         TEXT NOT NULL,
    provider             TEXT NOT NULL,
    -- payer omitted: single-payer assumption, plan distinction unused
    encounterclass       TEXT NOT NULL,
    code                 TEXT NOT NULL,
    description          TEXT NOT NULL,
    base_encounter_cost  DOUBLE PRECISION NOT NULL,
    total_claim_cost     DOUBLE PRECISION NOT NULL,
    reasoncode           TEXT,
    reasondescription    TEXT
);

INSERT INTO encounters
SELECT
    id,
    patient,
    CAST(CAST(start AS TIMESTAMP) AS DATE) AS start_date,   -- TZ-aware; resolve offset THEN date 
    CAST(CAST(stop  AS TIMESTAMP) AS DATE) AS stop_date,
    organization,
    provider,
    encounterclass,
    CAST(code AS TEXT),                        -- BIGINT identifier -> TEXT
    description,
    base_encounter_cost,
    total_claim_cost,
    CAST(reasoncode AS TEXT),                  -- BIGINT identifier -> TEXT
    reasondescription
FROM raw_encounters
WHERE CAST(CAST(start AS TIMESTAMP) AS DATE) <= DATE '2026-12-31';

SELECT * FROM encounters LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

id,patient,start_date,stop_date,organization,provider,encounterclass,code,description,base_encounter_cost,total_claim_cost,reasoncode,reasondescription
9a82dee1-75d3-ef90-6e5e-b6e6aab262ce,9a82dee1-75d3-ef90-6e2f-ab7254dd1069,2023-06-22,2023-06-22,cad0e59d-6171-33f8-90dd-eda95572f499,0073821d-cde7-3e63-b8fb-0fc98def0fd5,ambulatory,185347001,Encounter for problem (procedure),84.68,205.8,431857002,Chronic kidney disease stage 4 (disorder)
8dd8cc29-92e8-938c-65c8-f6c33ff01dd9,8dd8cc29-92e8-938c-7701-06514fecb7bc,2017-04-23,2017-04-23,b72e02fc-eb0e-3962-a3a8-6fe29c43ea26,410cdd1a-df31-31d8-a874-9e01eb8ccbaa,ambulatory,185347001,Encounter for problem (procedure),84.68,493.5,431857002,Chronic kidney disease stage 4 (disorder)
8dd8cc29-92e8-938c-731c-49784d8ce779,8dd8cc29-92e8-938c-7701-06514fecb7bc,2017-04-26,2017-04-26,b72e02fc-eb0e-3962-a3a8-6fe29c43ea26,410cdd1a-df31-31d8-a874-9e01eb8ccbaa,ambulatory,185347001,Encounter for problem (procedure),84.68,818.28,431857002,Chronic kidney disease stage 4 (disorder)
9a82dee1-75d3-ef90-bb02-7ac5c980a38c,9a82dee1-75d3-ef90-6e2f-ab7254dd1069,2023-06-25,2023-06-25,cad0e59d-6171-33f8-90dd-eda95572f499,0073821d-cde7-3e63-b8fb-0fc98def0fd5,ambulatory,185347001,Encounter for problem (procedure),84.68,493.5,431857002,Chronic kidney disease stage 4 (disorder)
9a82dee1-75d3-ef90-4c8d-3f433c563635,9a82dee1-75d3-ef90-6e2f-ab7254dd1069,2023-06-28,2023-06-28,cad0e59d-6171-33f8-90dd-eda95572f499,0073821d-cde7-3e63-b8fb-0fc98def0fd5,ambulatory,185347001,Encounter for problem (procedure),84.68,1565.4,431857002,Chronic kidney disease stage 4 (disorder)


### Conditions

In [30]:
%%sql

SELECT * FROM raw_conditions LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION
1967-09-08,None,a96f4412-e403-f107-78ce-8ca08d2863dc,a96f4412-e403-f107-4464-59069193524a,SNOMED-CT,203646004,Adolescent idiopathic scoliosis (disorder)
1975-09-01,None,a96f4412-e403-f107-78ce-8ca08d2863dc,a96f4412-e403-f107-aecc-df43cacf1e95,SNOMED-CT,224299000,Received higher education (finding)
1979-09-10,None,a96f4412-e403-f107-78ce-8ca08d2863dc,a96f4412-e403-f107-6e1f-f39351e8bb45,SNOMED-CT,714628002,Prediabetes (finding)
1979-09-10,None,a96f4412-e403-f107-78ce-8ca08d2863dc,a96f4412-e403-f107-6e1f-f39351e8bb45,SNOMED-CT,271737000,Anemia (disorder)
1976-02-24,None,726803cb-9079-e62d-ed4a-d89c5ecbb29b,726803cb-9079-e62d-335f-808659046c0e,SNOMED-CT,224295006,Only received primary school education (finding)


In [31]:
%%sql

SELECT *
FROM raw_conditions
GROUP BY ALL
HAVING count(*) > 1

Running query in 'duckdb:///../ma_quality.duckdb'

START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION


In [32]:
%%sql

SELECT column_name, column_type, min, max, approx_unique, count, null_percentage
FROM (SUMMARIZE raw_conditions)

Running query in 'duckdb:///../ma_quality.duckdb'

column_name,column_type,min,max,approx_unique,count,null_percentage
START,DATE,1941-03-26,2027-02-14,25371,2399532,0.00
STOP,DATE,2012-02-18,2027-02-14,5278,2399532,19.85
PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,2399532,0.00
ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-042a-fa27c5d83c1f,fff5352f-2b53-d86c-fdcf-8582350378f2,1438665,2399532,0.00
SYSTEM,VARCHAR,SNOMED-CT,SNOMED-CT,1,2399532,0.00
CODE,BIGINT,1734006,10939881000119105,270,2399532,0.00
DESCRIPTION,VARCHAR,Abnormal findings diagnostic imaging heart+coronary circulat (finding),Whiplash injury to neck (disorder),297,2399532,0.00


First stop in 2012 is a symptom of generation restrictions. I restricted the end date of conditions to be 15 years or less to limit the rows my computer had to process. COL-E measure needs the longest history of data at 10 years of patient history and the earliest year used in 2022 so I 15 years is all that is needed.

~20% nulls is plausible to be either lifelong or active conditions. Null is the correct representation for a condition ending

In [33]:
%%sql

SELECT
    CAST(code AS TEXT) AS code,
    description,
    count(*) AS n,
    count(*) FILTER (WHERE stop IS NULL) AS null_stop,
    round(100.0 * count(*) FILTER (WHERE stop IS NULL) / count(*), 1) AS pct_null
FROM raw_conditions
WHERE stop IS NULL
GROUP BY code, description
ORDER BY n DESC
LIMIT 40;

Running query in 'duckdb:///../ma_quality.duckdb'

code,DESCRIPTION,n,null_stop,pct_null
162864005,Body mass index 30+ - obesity (finding),24562,24562,100.0
160903007,Full-time employment (finding),20166,20166,100.0
274531002,Abnormal findings diagnostic imaging heart+coronary circulat (finding),19572,19572,100.0
414545008,Ischemic heart disease (disorder),19513,19513,100.0
714628002,Prediabetes (finding),19273,19273,100.0
73595000,Stress (finding),17447,17447,100.0
271737000,Anemia (disorder),17444,17444,100.0
314529007,Medication review due (situation),16550,16550,100.0
224299000,Received higher education (finding),16361,16361,100.0
59621000,Essential hypertension (disorder),14514,14514,100.0


In [34]:
%%sql

SELECT
    count(*) FILTER (WHERE stop IS NOT NULL AND stop < start) AS inverted_spans,
FROM raw_conditions;

Running query in 'duckdb:///../ma_quality.duckdb'

inverted_spans
0


In [ ]:
%%sql

CREATE OR REPLACE TABLE conditions (
    patient      TEXT , -- foreign keys
    encounter    TEXT ,       
    start_date   DATE NOT NULL,
    stop_date    DATE,                    
    code         TEXT NOT NULL,           -- SNOMED-CT identifier, BIGINT -> TEXT
    description  TEXT NOT NULL
    -- system omitted: constant 'SNOMED-CT' across all rows
);

INSERT INTO conditions
SELECT
    c.patient,
    c.encounter,
    CAST(CAST(c.start AS TIMESTAMP) AS DATE) AS start_date,
    CAST(CAST(c.stop  AS TIMESTAMP) AS DATE) AS stop_date,
    CAST(c.code AS TEXT),      
    c.description
FROM raw_conditions c;

SELECT * FROM conditions LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

patient,encounter,start_date,stop_date,code,description
81df9a0c-cd61-2a9d-b7ec-9c084e88d30a,81df9a0c-cd61-2a9d-550d-97646a056a9d,2016-02-10,2017-02-15,73595000,Stress (finding)
663a3f92-f9cd-fb5b-37f5-e857b11322c5,663a3f92-f9cd-fb5b-e320-92c02f958794,2014-02-06,None,82423001,Chronic pain (finding)
663a3f92-f9cd-fb5b-37f5-e857b11322c5,663a3f92-f9cd-fb5b-e320-92c02f958794,2014-02-06,None,278860009,Chronic low back pain (finding)
663a3f92-f9cd-fb5b-37f5-e857b11322c5,663a3f92-f9cd-fb5b-e320-92c02f958794,2014-02-06,None,1121000119107,Chronic neck pain (finding)
663a3f92-f9cd-fb5b-37f5-e857b11322c5,663a3f92-f9cd-fb5b-c5df-e12ee2787ccb,2014-03-25,2017-04-11,73595000,Stress (finding)


### Observations

In [36]:
%%sql 

SELECT * FROM raw_observations LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE
2012-04-30 15:29:33+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,vital-signs,8302-2,Body Height,170.6,cm,numeric
2012-04-30 15:29:33+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,vital-signs,72514-3,Pain severity - 0-10 verbal numeric rating [Score] - Reported,3.0,{score},numeric
2012-04-30 15:29:33+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,vital-signs,29463-7,Body Weight,88.1,kg,numeric
2012-04-30 15:29:33+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,vital-signs,39156-5,Body mass index (BMI) [Ratio],30.3,kg/m2,numeric
2012-04-30 15:29:33+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,vital-signs,8462-4,Diastolic Blood Pressure,71.0,mm[Hg],numeric


In [37]:
%%sql

SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_observations)) AS exact_dupes
FROM raw_observations

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

exact_dupes
20323


In [38]:
%%sql

SELECT date, patient, encounter, category, code, description, value, units, type, count(*) AS n
FROM raw_observations
GROUP BY date, patient, encounter, category, code, description, value, units, type
HAVING count(*) > 1
ORDER BY n DESC, patient, date

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE,n
2025-12-02 07:18:23+00:00,00058803-aca9-56cf-a3a8-8f22cd7e1c2d,00058803-aca9-56cf-0ece-231fa7a9be4d,laboratory,6298-4,Potassium [Moles/volume] in Blood,4.7,mmol/L,numeric,2
2025-12-02 07:18:23+00:00,00058803-aca9-56cf-a3a8-8f22cd7e1c2d,00058803-aca9-56cf-0ece-231fa7a9be4d,laboratory,6299-2,Urea nitrogen [Mass/volume] in Blood,9.9,mg/dL,numeric,2
2025-12-02 07:18:23+00:00,00058803-aca9-56cf-a3a8-8f22cd7e1c2d,00058803-aca9-56cf-0ece-231fa7a9be4d,laboratory,2947-0,Sodium [Moles/volume] in Blood,136.7,mmol/L,numeric,2
2025-12-02 07:18:23+00:00,00058803-aca9-56cf-a3a8-8f22cd7e1c2d,00058803-aca9-56cf-0ece-231fa7a9be4d,laboratory,2339-0,Glucose [Mass/volume] in Blood,74.6,mg/dL,numeric,2
2025-12-02 07:18:23+00:00,00058803-aca9-56cf-a3a8-8f22cd7e1c2d,00058803-aca9-56cf-0ece-231fa7a9be4d,laboratory,2069-3,Chloride [Moles/volume] in Blood,110.0,mmol/L,numeric,2
2025-12-02 07:18:23+00:00,00058803-aca9-56cf-a3a8-8f22cd7e1c2d,00058803-aca9-56cf-0ece-231fa7a9be4d,laboratory,20565-8,Carbon dioxide total [Moles/volume] in Blood,25.7,mmol/L,numeric,2
2025-12-02 07:18:23+00:00,00058803-aca9-56cf-a3a8-8f22cd7e1c2d,00058803-aca9-56cf-0ece-231fa7a9be4d,laboratory,49765-1,Calcium [Mass/volume] in Blood,10.0,mg/dL,numeric,2
2017-05-12 16:42:02+00:00,0014d898-243c-7301-9433-0964bef654d7,0014d898-243c-7301-4545-07f79dcfbd17,vital-signs,2708-6,Oxygen saturation in Arterial blood,91.0,%,numeric,2
2017-05-12 16:42:02+00:00,0014d898-243c-7301-9433-0964bef654d7,0014d898-243c-7301-4545-07f79dcfbd17,laboratory,2744-1,pH of Arterial blood,6.8,[pH],numeric,2
2017-07-19 15:09:36+00:00,002b4e5f-f303-c3fc-676e-a39e834892e6,002b4e5f-f303-c3fc-462e-1a4efcebf426,laboratory,49765-1,Calcium [Mass/volume] in Blood,9.3,mg/dL,numeric,2


20,323 exact duplicate columns that need to be dropped on insert keeping only one of the two.

In [39]:
%%sql

SELECT date, patient, encounter, category, code, description, value, units, type, count(*) AS n
FROM raw_observations
GROUP BY date, patient, encounter, category, code, description, value, units, type
HAVING count(*) > 2
ORDER BY n DESC, patient, date

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

DATE,PATIENT,ENCOUNTER,CATEGORY,CODE,DESCRIPTION,VALUE,UNITS,TYPE,n


Duplicates are due to pannel + component double emission where synthea generates two rows of the exact same information. Will be removed on insert.

In [40]:
%%sql

SELECT column_name, column_type, min, max, approx_unique, count, null_percentage
FROM (SUMMARIZE raw_observations)

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,count,null_percentage
DATE,TIMESTAMP WITH TIME ZONE,2011-01-01 09:59:43+00,2027-02-14 23:59:20+00,4850355,65408633,0.00
PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,65408633,0.00
ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-03ca-a9f8fbddd451,fff5352f-2b53-d86c-fdcf-8582350378f2,2046343,65408633,2.15
CATEGORY,VARCHAR,exam,vital-signs,7,65408633,2.15
CODE,VARCHAR,10230-1,X9999-2,296,65408633,0.00
DESCRIPTION,VARCHAR,A great deal of time is spent in activities necessary to obtain the opioid use the opioid or recover from its effects,pH of Venous blood,299,65408633,0.00
VALUE,VARCHAR,-0.0,urgent,52762,65408633,0.00
UNITS,VARCHAR,%,{score},48,65408633,27.39
TYPE,VARCHAR,numeric,text,2,65408633,0.00


In [41]:
%%sql

SELECT category, type,
       count(*) AS n,
       count(*) FILTER (WHERE encounter IS NULL) AS null_enc,
       round(100.0*count(*) FILTER (WHERE encounter IS NULL)/count(*),1) AS pct_null
FROM raw_observations
GROUP BY category, type
ORDER BY null_enc DESC

Running query in 'duckdb:///../ma_quality.duckdb'

CATEGORY,TYPE,n,null_enc,pct_null
None,numeric,1405098,1405098,100.0
exam,numeric,15576,0,0.0
vital-signs,numeric,8696411,0,0.0
survey,text,17748436,0,0.0
procedure,numeric,40203,0,0.0
survey,numeric,3355763,0,0.0
laboratory,numeric,25215114,0,0.0
procedure,text,12332,0,0.0
laboratory,text,7660893,0,0.0
exam,text,308645,0,0.0


In [42]:
%%sql

SELECT code, description, units, count(*) AS n
FROM raw_observations
WHERE encounter IS NULL
GROUP BY code, description, units
ORDER BY n DESC


Running query in 'duckdb:///../ma_quality.duckdb'

CODE,DESCRIPTION,UNITS,n
QALY,QALY,a,468366
QOLS,QOLS,{score},468366
DALY,DALY,a,468366


The nulls are book keeping metrics not clinical data making them nulls in encounter and category. Will be dropped on insert.

In [43]:
%%sql

SELECT code, count(DISTINCT description) AS n_desc
FROM raw_observations
GROUP BY code
HAVING count(DISTINCT description) > 1
ORDER BY n_desc DESC

Running query in 'duckdb:///../ma_quality.duckdb'

CODE,n_desc
6299-2,2


In [44]:
%%sql

SELECT code, description, count(*) AS n
FROM raw_observations
WHERE code = '6299-2'
GROUP BY code, description


Running query in 'duckdb:///../ma_quality.duckdb'

CODE,DESCRIPTION,n
6299-2,Urea nitrogen [Mass/volume] in Blood,862856
6299-2,Urea Nitrogen [Mass/volume] in Blood,18179


Normalize case sensitivity for codes on insert.

In [45]:
%%sql

SELECT type,
       count(*) AS n,
       count(*) FILTER (WHERE units IS NULL) AS null_units,
       round(100.0*count(*) FILTER (WHERE units IS NULL)/count(*),1) AS pct_null
FROM raw_observations
GROUP BY type

Running query in 'duckdb:///../ma_quality.duckdb'

TYPE,n,null_units,pct_null
text,26679067,17916469,67.2
numeric,38729566,0,0.0


Text observations are unitless - Null is the correct representation for absence.

In [46]:
%%sql

SELECT type, count(*) FROM raw_observations GROUP BY type

Running query in 'duckdb:///../ma_quality.duckdb'

TYPE,count_star()
text,26679067
numeric,38729566


In [ ]:
%%sql 

CREATE OR REPLACE TABLE observations (
    date         DATE NOT NULL,
    patient      TEXT, -- foreign keys
    encounter    TEXT,        
    category     TEXT NOT NULL,
    code         TEXT NOT NULL,        
    description  TEXT NOT NULL,
    value        TEXT NOT NULL,         
    units        TEXT,                  
    type         TEXT NOT NULL           
);

INSERT INTO observations
SELECT DISTINCT
    CAST(CAST(date AS TIMESTAMP) AS DATE),   
    o.patient,
    o.encounter,
    o.category,
    o.code,
    lower(o.description),                     
    o.value,
    o.units,
    o.type
FROM raw_observations o 
JOIN encounters e 
	on e.id = o.encounter
WHERE o.code NOT IN ('QALY', 'DALY', 'QOLS');

SELECT * FROM observations LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

date,patient,encounter,category,code,description,value,units,type
2014-01-07,dc6c070c-e428-280d-3763-44e8b805d63c,dc6c070c-e428-280d-a570-e654f5439ba9,survey,63512-8,how many people are living or staying at this address [#],8.0,{#},numeric
2024-08-14,3dd94c75-3a20-8fb4-eaee-89a6a56c0a7b,3dd94c75-3a20-8fb4-4798-506fc1e4e2e7,laboratory,33914-3,glomerular filtration rate [volume rate/area] in serum or plasma by creatinine-based formula (mdrd)/1.73 sq m,75.1,mL/min/{1.73_m2},numeric
2014-05-23,dc6c070c-e428-280d-3763-44e8b805d63c,dc6c070c-e428-280d-b8ce-11cbfcb64f69,vital-signs,74006-8,weight difference [mass difference] --pre dialysis - post dialysis,2.5,kg,numeric
2024-10-23,3dd94c75-3a20-8fb4-eaee-89a6a56c0a7b,3dd94c75-3a20-8fb4-7059-db9d181e5210,laboratory,5792-7,glucose [mass/volume] in urine by test strip,1.1,mg/dL,numeric
2014-09-15,dc6c070c-e428-280d-3763-44e8b805d63c,dc6c070c-e428-280d-e1a1-cd4ecbc126f1,laboratory,6298-4,potassium [moles/volume] in blood,4.9,mmol/L,numeric


### Medications

In [48]:
%%sql

SELECT * FROM raw_medications LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

START,STOP,PATIENT,PAYER,ENCOUNTER,CODE,DESCRIPTION,BASE_COST,PAYER_COVERAGE,DISPENSES,TOTALCOST,REASONCODE,REASONDESCRIPTION
1966-02-03 10:36:41+00:00,None,4ab6f786-0913-43c5-7121-afb3db14952a,df166300-5a78-3502-a46a-832842197811,4ab6f786-0913-43c5-2880-6932330e2df1,1049630,diphenhydrAMINE Hydrochloride 25 MG Oral Tablet,124.48,0.0,743,92488.64,None,None
1964-01-28 19:17:10+00:00,None,18acf270-d944-4228-ef95-e395bb4e0dba,e03e23c9-4df1-3eb6-a62d-f70f02301496,18acf270-d944-4228-f740-cf2cf36975cd,477045,Chlorpheniramine Maleate 2 MG/ML Oral Solution,124.48,0.0,767,95476.16,None,None
1998-02-15 23:06:44+00:00,None,a96f4412-e403-f107-78ce-8ca08d2863dc,e03e23c9-4df1-3eb6-a62d-f70f02301496,a96f4412-e403-f107-5c94-367eb0cbe79e,849574,Naproxen sodium 220 MG Oral Tablet,342.34,0.0,353,120846.02,201834006,Localized primary osteoarthritis of the hand (disorder)
2001-04-24 05:16:52+00:00,None,79075df7-96c4-4573-dd60-4828abaa9cc2,e03e23c9-4df1-3eb6-a62d-f70f02301496,79075df7-96c4-4573-11aa-64666ad24be9,309362,Clopidogrel 75 MG Oral Tablet,964.02,0.0,314,302702.28,None,None
2001-04-24 05:16:52+00:00,None,79075df7-96c4-4573-dd60-4828abaa9cc2,e03e23c9-4df1-3eb6-a62d-f70f02301496,79075df7-96c4-4573-11aa-64666ad24be9,312961,Simvastatin 20 MG Oral Tablet,171.09,0.0,314,53722.26,None,None


In [49]:
%%sql

SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_medications)) AS exact_dupes
FROM raw_medications

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

exact_dupes
0


In [50]:
%%sql

SELECT column_name, column_type, min, max, approx_unique, count, null_percentage
FROM (SUMMARIZE raw_medications)

Running query in 'duckdb:///../ma_quality.duckdb'

column_name,column_type,min,max,approx_unique,count,null_percentage
START,TIMESTAMP WITH TIME ZONE,1941-03-27 06:13:30+00,2027-02-14 23:59:20+00,1520571,4117058,0.00
STOP,TIMESTAMP WITH TIME ZONE,2012-02-17 09:46:17+00,2027-02-14 23:59:20+00,1421828,4117058,4.59
PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,4117058,0.00
PAYER,VARCHAR,0133f751-9229-3cfd-815f-b6d4979bdd6a,e03e23c9-4df1-3eb6-a62d-f70f02301496,10,4117058,0.00
ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-03ca-a9f8fbddd451,fff5352f-2b53-d86c-fdcf-8582350378f2,1936353,4117058,0.00
CODE,BIGINT,105585,2563431,286,4117058,0.00
DESCRIPTION,VARCHAR,0.25 ML Leuprolide Acetate 30 MG/ML Prefilled Syringe,{7 (ethinyl estradiol 0.035 MG / norgestimate 0.18 MG Oral Tablet) / 7 (ethinyl estradiol 0.035 MG / norgestimate 0.215 MG Oral Tablet) / 7 (ethinyl estradiol 0.035 MG / norgestimate 0.25 MG Oral Tablet) / 7 (inert ingredients 1 MG Oral Tablet) } Pack [Trinessa 28 Day],507,4117058,0.00
BASE_COST,DOUBLE,0.01,211629.6,140691,4117058,0.00
PAYER_COVERAGE,DOUBLE,0.0,211629.6,102598,4117058,0.00
DISPENSES,BIGINT,1,1045,1179,4117058,0.00


Nulls in the stop, reasoncode and reason description are expected and the correct representation for absence.

In [51]:
%%sql

SELECT
    count(*) FILTER (WHERE stop IS NULL)                                   AS active_rx,
    count(*) FILTER (WHERE stop IS NOT NULL
                       AND CAST(stop  AT TIME ZONE 'UTC' AS DATE)
                         < CAST(start AT TIME ZONE 'UTC' AS DATE))         AS inverted_spans
FROM raw_medications;

Running query in 'duckdb:///../ma_quality.duckdb'

active_rx,inverted_spans
189125,5440


In [52]:
%%sql
SELECT patient, CAST(code AS TEXT) AS code,
       start, stop,
       CAST(start AT TIME ZONE 'UTC' AS DATE) AS start_d,
       CAST(stop  AT TIME ZONE 'UTC' AS DATE) AS stop_d,
       stop - start AS interval_raw
FROM raw_medications
WHERE stop IS NOT NULL
  AND CAST(stop AT TIME ZONE 'UTC' AS DATE) < CAST(start AT TIME ZONE 'UTC' AS DATE)


Running query in 'duckdb:///../ma_quality.duckdb'

PATIENT,code,START,STOP,start_d,stop_d,interval_raw
c42903d7-2d39-9f41-cb69-722b2251c9d2,308136,2023-05-03 21:52:51+00:00,2023-04-28 21:52:51+00:00,2023-05-03,2023-04-28,"-5 days, 0:00:00"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,106892,2012-06-03 19:52:16+00:00,2012-05-28 19:54:32+00:00,2012-06-03,2012-05-28,"-6 days, 0:02:16"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,860975,2012-06-03 19:52:16+00:00,2012-05-28 19:54:32+00:00,2012-06-03,2012-05-28,"-6 days, 0:02:16"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,314076,2012-06-03 19:52:16+00:00,2012-05-28 19:54:32+00:00,2012-06-03,2012-05-28,"-6 days, 0:02:16"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,308136,2012-06-03 19:52:16+00:00,2012-05-28 19:54:32+00:00,2012-06-03,2012-05-28,"-6 days, 0:02:16"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,106892,2014-06-15 19:52:16+00:00,2014-06-09 20:24:17+00:00,2014-06-15,2014-06-09,"-6 days, 0:32:01"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,860975,2014-06-15 19:52:16+00:00,2014-06-09 20:24:17+00:00,2014-06-15,2014-06-09,"-6 days, 0:32:01"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,314076,2014-06-15 19:52:16+00:00,2014-06-09 20:24:17+00:00,2014-06-15,2014-06-09,"-6 days, 0:32:01"
7931444a-bfdf-a287-e91b-3074b9dd0fe1,308136,2014-06-15 19:52:16+00:00,2014-06-09 20:24:17+00:00,2014-06-15,2014-06-09,"-6 days, 0:32:01"
1a1a1f42-71a3-1bbd-145b-c64fe3be1beb,310798,2013-11-24 22:07:42+00:00,2013-11-19 22:07:42+00:00,2013-11-24,2013-11-19,"-5 days, 0:00:00"


In [53]:
%%sql
SELECT
    count(*) FILTER (WHERE stop IS NOT NULL AND stop < start) AS inverted_timestamp,
    count(*) FILTER (WHERE stop IS NOT NULL
                       AND CAST(stop AT TIME ZONE 'UTC' AS DATE) < CAST(start AT TIME ZONE 'UTC' AS DATE)) AS inverted_datecast
FROM raw_medications;

Running query in 'duckdb:///../ma_quality.duckdb'

inverted_timestamp,inverted_datecast
5440,5440


In [54]:
%%sql

SELECT patient, code, start, stop, count(*) AS n
FROM raw_medications
WHERE stop IS NOT NULL
  AND CAST(stop AT TIME ZONE 'UTC' AS DATE) < CAST(start AT TIME ZONE 'UTC' AS DATE)
GROUP BY patient, code, start, stop
HAVING count(*) > 1
ORDER BY n DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

PATIENT,CODE,START,STOP,n


In [55]:
%%sql

SELECT CAST(code AS TEXT) AS code, count(DISTINCT description) AS n_desc
FROM raw_medications
GROUP BY code
HAVING count(DISTINCT description) > 1
ORDER BY n_desc DESC


Running query in 'duckdb:///../ma_quality.duckdb'

code,n_desc
1049221,2
1659149,2
855332,2
312961,2
1804799,2
1437975,2
1000126,2
309362,2
979485,2
583214,2


In [56]:
%%sql

SELECT CAST(code AS TEXT) AS code,
       count(DISTINCT description)          AS distinct_desc,
       count(DISTINCT lower(description))   AS distinct_desc_lower
FROM raw_medications
GROUP BY code
HAVING count(DISTINCT description) > 1
ORDER BY distinct_desc DESC;

Running query in 'duckdb:///../ma_quality.duckdb'

code,distinct_desc,distinct_desc_lower
245314,2,1
1437975,2,1
1000126,2,1
1659149,2,1
979485,2,1
309362,2,1
856987,2,1
1049221,2,1
855332,2,1
314231,2,1


Duplicate descriptions are due to case sensitivity and need to be normalized on insert.

In [57]:
%%sql

SELECT
    count(*) FILTER (WHERE base_cost < 0 OR totalcost < 0 OR dispenses < 0)          AS negatives,
    count(*) FILTER (WHERE round(base_cost * dispenses, 2) <> round(totalcost, 2))   AS cost_identity_breaks
FROM raw_medications

Running query in 'duckdb:///../ma_quality.duckdb'

negatives,cost_identity_breaks
0,0


In [58]:
%%sql

SELECT
    count(*) FILTER (WHERE (reasoncode IS NULL) <> (reasondescription IS NULL)) AS mismatched,
    round(100.0*count(*) FILTER (WHERE reasoncode IS NULL)/count(*),1)          AS pct_null
FROM raw_medications

Running query in 'duckdb:///../ma_quality.duckdb'

mismatched,pct_null
0,17.2


In [ ]:
%%sql

CREATE OR REPLACE TABLE medications_quarantine AS
SELECT *, 'inverted_span_stop_before_start' AS quarantine_reason
FROM raw_medications
WHERE stop IS NOT NULL
  AND stop < start;

CREATE OR REPLACE TABLE medications (
    patient      TEXT, -- foreign keys
    encounter    TEXT,           
    start_date   DATE NOT NULL,
    stop_date    DATE,                      
    code         TEXT NOT NULL,            
    description  TEXT NOT NULL,             
    base_cost    DOUBLE PRECISION NOT NULL,
    dispenses    BIGINT NOT NULL,
    totalcost    DOUBLE PRECISION NOT NULL,
    -- payer, payer_coverage omitted: single-payer assumption
    reasoncode         TEXT,                
    reasondescription  TEXT
);

INSERT INTO medications
SELECT
    m.patient,
    m.encounter,
    CAST(CAST(m.start AS TIMESTAMP) AS DATE),  
    CAST(CAST(m.stop  AS TIMESTAMP) AS DATE),
    CAST(m.code AS TEXT),
    lower(m.description),                        
    m.base_cost,
    m.dispenses,
    m.totalcost,
    CAST(m.reasoncode AS TEXT),
    m.reasondescription
FROM raw_medications m
JOIN encounters e 
	ON e.id = m.encounter
WHERE m.start <= m.stop OR m.stop IS NULL;

SELECT * FROM medications_quarantine LIMIT 5;

SELECT * FROM medications LIMIT 5;

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

patient,encounter,start_date,stop_date,code,description,base_cost,dispenses,totalcost,reasoncode,reasondescription
4ab6f786-0913-43c5-7121-afb3db14952a,4ab6f786-0913-43c5-2880-6932330e2df1,1966-02-03,None,1049630,diphenhydramine hydrochloride 25 mg oral tablet,124.48,743,92488.64,None,None
18acf270-d944-4228-ef95-e395bb4e0dba,18acf270-d944-4228-f740-cf2cf36975cd,1964-01-28,None,477045,chlorpheniramine maleate 2 mg/ml oral solution,124.48,767,95476.16,None,None
a96f4412-e403-f107-78ce-8ca08d2863dc,a96f4412-e403-f107-5c94-367eb0cbe79e,1998-02-15,None,849574,naproxen sodium 220 mg oral tablet,342.34,353,120846.02,201834006,Localized primary osteoarthritis of the hand (disorder)
79075df7-96c4-4573-dd60-4828abaa9cc2,79075df7-96c4-4573-11aa-64666ad24be9,2001-04-24,None,705129,nitroglycerin 0.4 mg/actuat mucosal spray,937.27,314,294302.78,None,None
79075df7-96c4-4573-dd60-4828abaa9cc2,79075df7-96c4-4573-1586-168543fa0f53,2011-06-15,2012-06-20,310798,hydrochlorothiazide 25 mg oral tablet,1.86,4,7.44,59621000,Essential hypertension (disorder)


### Procedures

In [60]:
%%sql 

SELECT * FROM raw_procedures LIMIT 5

Running query in 'duckdb:///../ma_quality.duckdb'

START,STOP,PATIENT,ENCOUNTER,SYSTEM,CODE,DESCRIPTION,BASE_COST,REASONCODE,REASONDESCRIPTION
2012-04-30 15:29:33+00:00,2012-04-30 16:11:53+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,SNOMED-CT,710824005,Assessment of health and social care needs (procedure),370.2,None,None
2012-04-30 16:11:53+00:00,2012-04-30 16:31:40+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,SNOMED-CT,710841007,Assessment of anxiety (procedure),370.2,None,None
2012-04-30 16:31:40+00:00,2012-04-30 16:42:18+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,SNOMED-CT,171207006,Depression screening (procedure),370.2,None,None
2012-04-30 16:42:18+00:00,2012-04-30 17:06:46+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,SNOMED-CT,171207006,Depression screening (procedure),370.2,None,None
2012-04-30 17:06:46+00:00,2012-04-30 17:21:14+00:00,7ecfde43-6f06-5da9-d6db-05fba4c24340,7ecfde43-6f06-5da9-c7cb-6f2e68ab1028,SNOMED-CT,428211000124100,Assessment of substance use (procedure),370.2,None,None


In [61]:
%%sql

SELECT count(*) - (SELECT count(*) FROM (SELECT DISTINCT * FROM raw_procedures)) AS duplicates
FROM raw_procedures

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicates
0


In [62]:
%%sql

SELECT column_name, column_type, min, max, approx_unique, count, null_percentage
FROM (SUMMARIZE raw_procedures)

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

column_name,column_type,min,max,approx_unique,count,null_percentage
START,TIMESTAMP WITH TIME ZONE,2012-02-01 21:14:45+00,2027-02-14 23:59:20+00,7984959,11672627,0.00
STOP,TIMESTAMP WITH TIME ZONE,2012-02-19 00:24:02+00,2027-02-18 17:30:55+00,9438428,11672627,0.00
PATIENT,VARCHAR,0003b2f9-73d5-3dcd-f7d5-1405661ebf67,fff5352f-2b53-d86c-2b2a-8f1e62b5be16,28979,11672627,0.00
ENCOUNTER,VARCHAR,0003b2f9-73d5-3dcd-0098-a8a6ec394d02,fff5352f-2b53-d86c-fdcf-8582350378f2,2453020,11672627,0.00
SYSTEM,VARCHAR,SNOMED-CT,SNOMED-CT,1,11672627,0.00
CODE,BIGINT,3457005,16335031000119103,532,11672627,0.00
DESCRIPTION,VARCHAR,12 lead electrocardiogram (procedure),Weaning from mechanically assisted ventilation (regime/therapy),417,11672627,0.00
BASE_COST,DOUBLE,0.28,260035.95,207193,11672627,0.00
REASONCODE,BIGINT,1734006,368581000119106,91,11672627,60.76
REASONDESCRIPTION,VARCHAR,Abnormal findings diagnostic imaging heart+coronary circulat (finding),Tubal pregnancy (disorder),101,11672627,60.76


In [63]:
%%sql

SELECT
    count(*) AS total,
    count(*) FILTER (WHERE REASONCODE IS NULL)         AS null_code,
    count(*) FILTER (WHERE REASONDESCRIPTION IS NULL)  AS null_desc,
    count(*) FILTER (WHERE REASONCODE IS NULL
                       AND REASONDESCRIPTION IS NULL)  AS both_null,
    count(*) FILTER (WHERE (REASONCODE IS NULL)
                        != (REASONDESCRIPTION IS NULL)) AS mismatch
FROM raw_procedures;

Running query in 'duckdb:///../ma_quality.duckdb'

total,null_code,null_desc,both_null,mismatch
11672627,7092586,7092586,7092586,0


In [64]:
%%sql

SELECT CODE,
       count(DISTINCT DESCRIPTION) AS n_desc
FROM raw_procedures
GROUP BY CODE
HAVING count(DISTINCT DESCRIPTION) > 1
ORDER BY n_desc DESC
LIMIT 25;

Running query in 'duckdb:///../ma_quality.duckdb'

CODE,n_desc


In [65]:
%%sql

SELECT REASONCODE,
       count(DISTINCT REASONDESCRIPTION) AS n_desc
FROM raw_procedures
WHERE REASONCODE IS NOT NULL
GROUP BY REASONCODE
HAVING count(DISTINCT REASONDESCRIPTION) > 1
ORDER BY n_desc DESC
LIMIT 25;

Running query in 'duckdb:///../ma_quality.duckdb'

REASONCODE,n_desc


Null codes-reasoncodes are matched and null by design

In [66]:
%%sql

SELECT count(*) AS inverted_spans
FROM raw_procedures
WHERE STOP < START;

Running query in 'duckdb:///../ma_quality.duckdb'

inverted_spans
6


In [ ]:
%%sql 

CREATE OR REPLACE TABLE procedures_quarantine AS
SELECT *, 'inverted_span_stop_before_start' AS quarantine_reason
FROM raw_procedures
WHERE stop < start;  

CREATE OR REPLACE TABLE procedures (
    patient            TEXT, 
    encounter          TEXT,
    start_date         DATE NOT NULL,
    stop_date          DATE NOT NULL,      
    code               TEXT NOT NULL,      
    description        TEXT NOT NULL,
    base_cost          DOUBLE PRECISION NOT NULL,
    -- system omitted: constant 'SNOMED-CT'
    reasoncode         TEXT,              
    reasondescription  TEXT
);

INSERT INTO procedures
SELECT
    p.patient,
    p.encounter,
    CAST(CAST(p.start AS TIMESTAMP) AS DATE),   
    CAST(CAST(p.stop  AS TIMESTAMP) AS DATE),
    CAST(p.code AS TEXT),
    p.description,
    p.base_cost,
    CAST(p.reasoncode AS TEXT),
    p.reasondescription
FROM raw_procedures p
WHERE p.start <= p.stop

Running query in 'duckdb:///../ma_quality.duckdb'

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Count


# Summary

Patients 
- Investigate the 174 null fips all in saylorville, polk county: Casted to null for now since it is an insignificant proportion of the population
- Dropped deathdate column: 100% null because the data was generated with the restriction that every patient is alive through the end of the generation

Encounters
- Dropped payers column: using the assumption that our whole population is covered by our medicare advantage plan this column becomes meaningless
- start_date and stop_date are casted to date from timestamp
- code and reasoncode are casted to text from bigint because they are identifier columns
- cast stop and start to date and renamed them to stop_date, start_date from stop, start

Conditions
- Dropped system column because it only has one value SNOMET-CT
- Casted code to text from bigint
- renamed start, stop columns to start_date, stop_date

Observations
- Exact duplicates are due to pannel + component double emission where synthea generates two rows of the exact same information. One row is kept and the other is filtered out on insert
- Case error code 'Urea Nitrogen [Mass/volume] in Blood' has an instance of an uppercase n in nitrogen and lower case n. Normalized to lower on insert
- The nulls in encounter and category are book keeping metrics not clinical data making them nulls in encounter and category. Not relavant to the analysis and are dropped on insert.

Medications
- 5540 rows have stop < start. rows will be quarentined to medications_quarentine table and dropped on insert into medications.
- Duplicate descriptions are due to case sensitivity so they are normalized on insert.
- Dropped payers column: using the assumption that our whole population is covered by our medicare advantage plan this column becomes meaningless
- Cast stop and start to date and renamed them to stop_date, start_date from stop, start
- Cast reasoncode to text from bigint because it is an identifier column

Procedures
- 6 rows have stop <. rows will be quarentined to procedures_quarentine table and dropped on insert into procedures
- Type casted code to text from bigint because it is an identifier column
- Dropped system column because it only has one value SNOMET-CT
- Cast stop and start to date and renamed them to stop_date, start_date from stop, start